# S02: Cointegration base rate of intraday pairs

Pre-registration: [`S02_README.md`](S02_README.md). Gates were fixed before data was downloaded and are not changed here.

**Question:** are same-sector pairs (and stock-vs-sector-ETF residuals) cointegrated more often than their empirical nulls, and does a formation-window pass predict a stationary frozen-β spread in the next 20 trading days?

**Data:** Alpaca SIP, 2020–2025, regular session. 2026 is sealed.

In [ ]:
import json, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))

from equity_trading import __version__
from equity_trading.alpaca import (ALPACA_VERSION, coverage, download_years, regular_session_panel,
                                   synthetic_panel)
from equity_trading.coint import COINT_VERSION, build_pairs, run_rolling
from equity_trading.data import work_dir
from equity_trading.s02 import S02_VERSION, adf_lift, describe, g1, g2, pass_rate_over_time, verdict
from equity_trading.universe import SECTOR_ETF_01, UNIVERSE_01

SYNTHETIC = os.environ.get("S02_SYNTHETIC") == "1"
YEARS = list(range(2020, 2026))                 # 2026 sealed
TIMEFRAMES = ["30Min", "15Min"]                 # 30Min primary
FORM_DAYS = [20, 60, 120]                       # 60 primary
STEP_DAYS, OOS_DAYS = 20, 20
PRIMARY = ("30Min", 60)
N_BOOT = 2000

OUT = ROOT / ("results/s02_synthetic" if SYNTHETIC else "results/s02")
OUT.mkdir(parents=True, exist_ok=True)
BIG = work_dir() / "results" / ("s02_synthetic" if SYNTHETIC else "s02")
BIG.mkdir(parents=True, exist_ok=True)

STOCKS = [s for v in UNIVERSE_01.values() for s in v]
ETFS = list(SECTOR_ETF_01.values())
SYMBOLS = STOCKS + ETFS
PAIRS = build_pairs(UNIVERSE_01, SECTOR_ETF_01)

# chart palette: fixed categorical order, one colour per group
COL = {"sector": "#2a78d6", "cross": "#eb6834", "etf": "#1baf7a", "etf_cross": "#eda100"}
GROUPS = list(COL)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.6,
                     "axes.edgecolor": "#9a9a95", "axes.titleweight": "bold", "font.size": 9})
print("synthetic:", SYNTHETIC, "| pairs:", PAIRS.group.value_counts().to_dict())

## 1. Data

Downloads only symbol-years missing from the cache, so it is safe to re-run.

In [ ]:
t0 = time.time()
panels = {}
for tf in TIMEFRAMES:
    if SYNTHETIC:
        panels[tf] = synthetic_panel(UNIVERSE_01, SECTOR_ETF_01, YEARS, bars_per_day=13 if tf == "30Min" else 26, seed=7)
    else:
        download_years(SYMBOLS, tf, YEARS)
        panels[tf] = regular_session_panel(SYMBOLS, tf, YEARS)
    print(tf, panels[tf].shape, f"{time.time() - t0:.0f}s")

In [ ]:
cov = coverage(panels["30Min"])
print("sessions per symbol (30Min):", cov["sessions"].min(), "-", cov["sessions"].max())
display(cov.sort_values("nan_share", ascending=False).head(8))
bars_per_session = panels["30Min"].notna().groupby(panels["30Min"].index.date).sum().median()
print("median regular-session bars per day (30Min), first symbols:", bars_per_session.head(5).to_dict())

In [ ]:
# Normalised daily closes by sector: stocks grey, sector ETF highlighted
daily = panels["30Min"].groupby(panels["30Min"].index.date).last()
daily.index = pd.to_datetime(daily.index)
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for ax, (sec, names) in zip(axes.flat, UNIVERSE_01.items()):
    base = daily[names + [SECTOR_ETF_01[sec]]].bfill().iloc[0]
    norm = daily[names + [SECTOR_ETF_01[sec]]] / base
    for s in names:
        ax.plot(norm.index, norm[s], color="#b5b5ae", lw=0.8)
    ax.plot(norm.index, norm[SECTOR_ETF_01[sec]], color=COL["sector"], lw=2, label=SECTOR_ETF_01[sec])
    ax.set_yscale("log"); ax.set_title(f"{sec} (grey: stocks)"); ax.legend(loc="upper left", frameon=False)
fig.suptitle("Price relative to first session, log scale"); fig.tight_layout(); plt.show()

## 2. Rolling Engle–Granger, every configuration

In [ ]:
results = {}
for tf in TIMEFRAMES:
    logp = np.log(panels[tf])
    for L in FORM_DAYS:
        t0 = time.time()
        res = run_rolling(logp, PAIRS, form_days=L, step_days=STEP_DAYS, oos_days=OOS_DAYS)
        res.to_parquet(BIG / f"rolling_{tf}_{L}.parquet", index=False)
        results[(tf, L)] = res
        print(f"{tf} L={L}: {len(res):,} pair-windows, {res.window.nunique()} windows, {time.time() - t0:.0f}s")

## 3. Gates (pre-registered)

In [ ]:
rows, gate_detail = [], {}
for (tf, L), res in results.items():
    for grp, nul in (("sector", "cross"), ("etf", "etf_cross")):
        a, b = g1(res, grp, nul, n_boot=N_BOOT), g2(res, grp, nul, n_boot=N_BOOT)
        c = adf_lift(res, grp, n_boot=N_BOOT)
        v = verdict(a, b)
        gate_detail[f"{tf}_{L}_{grp}"] = {"G1": a, "G2": b, "adf_lift_descriptive": c, "verdict": v}
        rows.append({"bars": tf, "L": L, "group": grp,
                     "pass_rate": a["rate_a"], "null_rate": a["rate_b"], "ratio": a["ratio"],
                     "G1_diff_lo": a["diff_lo"], "G1": a["pass"],
                     "band_hold_passers": b["rate_a"], "band_hold_null_passers": b["rate_b"],
                     "G2_lift": b["diff"], "G2_lift_lo": b["diff_lo"], "G2": b["pass"],
                     "adf_oos_lift": c["diff"], "verdict": v})
gates = pd.DataFrame(rows)
gates.to_csv(OUT / "gates_all_configs.csv", index=False, float_format="%.4f")

prim = gate_detail[f"{PRIMARY[0]}_{PRIMARY[1]}_sector"]
sec = gate_detail[f"{PRIMARY[0]}_{PRIMARY[1]}_etf"]
print(f"PRIMARY (sector vs cross, {PRIMARY[0]}, L={PRIMARY[1]}):  {prim['verdict']}")
print(f"SECONDARY (etf vs etf_cross, same config):            {sec['verdict']}")
gates.style.format({c: "{:.3f}" for c in gates.select_dtypes("float").columns})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, tf in zip(axes, TIMEFRAMES):
    w = 0.2
    for j, grp in enumerate(GROUPS):
        x = np.arange(len(FORM_DAYS)) + (j - 1.5) * w
        vals, lo, hi = [], [], []
        for L in FORM_DAYS:
            key = "sector" if grp in ("sector", "cross") else "etf"
            d = gate_detail[f"{tf}_{L}_{key}"]["G1"]
            k = "rate_a" if grp in ("sector", "etf") else "rate_b"
            vals.append(d[k]); lo.append(d[k] - d[k + "_lo"]); hi.append(d[k + "_hi"] - d[k])
        ax.bar(x, vals, w * 0.9, color=COL[grp], label=grp, yerr=[lo, hi], capsize=2, ecolor="#555550")
    ax.axhline(0.05, color="#555550", ls="--", lw=1)
    ax.text(-0.48, 0.055, "nominal 5%", ha="left", va="bottom", fontsize=8, color="#555550")
    ax.set_xticks(range(len(FORM_DAYS)), [f"L = {L} days" for L in FORM_DAYS]); ax.set_title(f"{tf} bars")
axes[0].set_ylabel("share of pair-windows with EG p ≤ 0.05"); axes[0].legend(frameon=False, ncol=2)
fig.suptitle("G1: formation pass rate by group (95% block-bootstrap CI)"); fig.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
pr = pass_rate_over_time(results[PRIMARY])
for grp in GROUPS:
    ax.plot(pr.index, pr[grp], color=COL[grp], lw=2, label=grp)
ax.axhline(0.05, color="#555550", ls="--", lw=1)
ax.set_ylabel("pass rate"); ax.legend(frameon=False, ncol=4, loc="upper left")
ax.set_title(f"Pass rate per formation window over time ({PRIMARY[0]}, L = {PRIMARY[1]})")
fig.tight_layout(); plt.show()
for (tf, L), res in results.items():
    pass_rate_over_time(res).to_csv(OUT / f"pass_rate_over_time_{tf}_{L}.csv", float_format="%.4f")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, tf in zip(axes, TIMEFRAMES):
    w = 0.38
    for j, (grp, nul) in enumerate([("sector", "cross"), ("etf", "etf_cross")]):
        b = gate_detail[f"{tf}_{PRIMARY[1]}_{grp}"]["G2"]
        for k, (key, g) in enumerate([("rate_a", grp), ("rate_b", nul)]):
            x = j + (k - 0.5) * w
            ax.bar(x, b[key], w * 0.9, color=COL[g],
                   yerr=[[b[key] - b[key + "_lo"]], [b[key + "_hi"] - b[key]]], capsize=2, ecolor="#555550")
            ax.text(x, -0.03, g, ha="center", va="top", fontsize=8)
    ax.set_xticks([0, 1], ["", ""]); ax.set_ylim(0, 1)
    ax.set_title(f"{tf}, L = {PRIMARY[1]}")
axes[0].set_ylabel("OOS share of bars within ±2 sd of formation mean")
fig.suptitle("G2: among formation passers, does the equilibrium hold for the next 20 days?")
fig.tight_layout(); plt.show()

## 4. Descriptives

In [ ]:
res = results[PRIMARY]
desc = describe(res)
for (tf, L), r in results.items():
    describe(r).to_csv(OUT / f"describe_{tf}_{L}.csv", float_format="%.4f")
display(desc.style.format("{:.3f}"))

bars_per_day = 13 if PRIMARY[0] == "30Min" else 26
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
s = res[res.group == "sector"]
ps = s.p <= 0.05
bins = np.linspace(0, 6, 41)
axes[0].hist(s.loc[~ps, "drift_sd"].clip(upper=6), bins, density=True, color="#b5b5ae", label="formation failed")
axes[0].hist(s.loc[ps, "drift_sd"].clip(upper=6), bins, density=True, color=COL["sector"], alpha=0.75, label="formation passed")
axes[0].set_xlabel("|OOS mean − formation mean| / formation sd (clipped at 6)"); axes[0].legend(frameon=False)
axes[0].set_title("Sector pairs: OOS equilibrium drift")
hl = (s.loc[ps, "half_life_bars"] / bars_per_day).clip(upper=30)
axes[1].hist(hl.dropna(), np.linspace(0, 30, 31), color=COL["sector"])
axes[1].set_xlabel("AR(1) half-life in formation, trading days (clipped at 30)")
axes[1].set_title("Sector pairs that passed: half-life")
fig.tight_layout(); plt.show()

In [ ]:
sect_of = {s: k for k, v in UNIVERSE_01.items() for s in v}
by_sector = (res[res.group.isin(["sector", "etf"])].assign(hit=lambda d: d.p <= 0.05, sector=lambda d: d.y.map(sect_of))
             .groupby(["sector", "group"])["hit"].mean().unstack("group"))
by_sector.to_csv(OUT / "pass_rate_by_sector.csv", float_format="%.4f")
by_sector.style.format("{:.3f}")

## 5. Save summary

In [ ]:
summary = {"study": "S02", "synthetic": SYNTHETIC, "years": YEARS, "primary": {"bars": PRIMARY[0], "L": PRIMARY[1]},
           "verdict_primary_sector": prim["verdict"], "verdict_secondary_etf": sec["verdict"],
           "gates": gate_detail,
           "versions": {"package": __version__, "alpaca": ALPACA_VERSION, "coint": COINT_VERSION, "s02": S02_VERSION},
           "n_boot": N_BOOT, "step_days": STEP_DAYS, "oos_days": OOS_DAYS,
           "data_end": str(panels["30Min"].index.max())}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
print(json.dumps({k: summary[k] for k in ["verdict_primary_sector", "verdict_secondary_etf", "data_end"]}, indent=2))